# 🚀 Fine-Tuning Mistral-7B for Python Code Generation using QLoRA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
[![HuggingFace Model](https://img.shields.io/badge/🤗-Model%20on%20Hub-yellow)](https://huggingface.co/navnani12/mistral-code-lora)

## 📌 Project Overview
This notebook demonstrates how to fine-tune **Mistral-7B**, a state-of-the-art large language model, to generate Python code from natural language instructions.

### 🎯 Goal
Given a natural language instruction like:
> *"Write a Python function to check if a number is prime"*

The fine-tuned model generates correct, working Python code.

### 🧠 Key Technique: QLoRA
Training a 7B parameter model normally requires **40+ GB of GPU memory** — way beyond what's available for free. We use **QLoRA (Quantized Low-Rank Adaptation)**:
- **4-bit quantization** compresses the model to fit in ~6GB GPU memory
- **LoRA adapters** train only 0.58% of parameters (42M out of 7.2B)
- Result: Full fine-tuning quality at a fraction of the cost

### 📊 Dataset
- **Source:** `iamtarun/python_code_instructions_18k_alpaca` on HuggingFace
- **Size:** 18,612 Python coding instruction-output pairs
- **Format:** Instruction → Python code

### 🛠️ Tech Stack
| Component | Tool |
|-----------|------|
| Base Model | Mistral-7B-v0.1 |
| Fine-tuning | QLoRA (PEFT + bitsandbytes) |
| Training | HuggingFace TRL (SFTTrainer) |
| Experiment Tracking | Weights & Biases |
| Model Hosting | HuggingFace Hub |
| Hardware | Free T4 GPU (Google Colab) |

---
⚡ **Runtime:** Make sure to enable GPU — `Runtime → Change runtime type → T4 GPU`

## 📦 Step 1: Install Dependencies

We install all required libraries:
- `transformers` — HuggingFace library for loading and using LLMs
- `datasets` — for loading training data from HuggingFace Hub
- `peft` — Parameter-Efficient Fine-Tuning (provides LoRA implementation)
- `accelerate` — handles distributed training and device management
- `bitsandbytes` — enables 4-bit quantization (the 'Q' in QLoRA)
- `trl` — Transformer Reinforcement Learning library with SFTTrainer
- `wandb` — Weights & Biases for experiment tracking

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl wandb huggingface_hub pyarrow --upgrade

## 🔐 Step 2: Authentication

We authenticate with two services:

1. **HuggingFace Hub** — to download the Mistral-7B model and push our fine-tuned adapter
   - Get your token at: https://huggingface.co/settings/tokens (needs Write permission)
   
2. **Weights & Biases** — to track training metrics (loss curves, GPU usage)
   - Get your API key at: https://wandb.ai/settings

In [ ]:
from huggingface_hub import login
import wandb

# HuggingFace login — paste your hf_xxxxx token
login(token="your_hf_token_here")

# Weights & Biases login — will prompt for API key
wandb.login()

## 📂 Step 3: Load the Dataset

We use the `iamtarun/python_code_instructions_18k_alpaca` dataset which contains **18,612 Python coding examples** in the Alpaca instruction format:

```
{
  "instruction": "Create a function to calculate the sum of a sequence",
  "input": "[1, 2, 3, 4, 5]",
  "output": "def sum_sequence(sequence): ..."
}
```

This format is ideal for instruction-following fine-tuning — the model learns to map natural language descriptions to working Python code.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split="train")
print(f"Dataset size: {len(dataset)}")
print(f"\nFeatures: {dataset.features}")
print(f"\nSample entry:")
print(f"Instruction: {dataset[0]['instruction']}")
print(f"Input: {dataset[0]['input']}")
print(f"Output: {dataset[0]['output'][:200]}")

## 🤖 Step 4: Load Mistral-7B in 4-bit (QLoRA Setup)

### Why 4-bit Quantization?
Mistral-7B in full precision (float32) requires ~28GB of GPU memory. A free Colab T4 GPU only has 16GB. By quantizing to 4-bit:
- Model size drops from ~28GB → ~4GB
- We can fit the model AND training overhead in 16GB

### BitsAndBytes Config:
- `load_in_4bit=True` — load weights in 4-bit precision
- `bnb_4bit_quant_type="nf4"` — NormalFloat4, best for normally-distributed weights
- `bnb_4bit_compute_dtype=float16` — use float16 for actual computations
- `bnb_4bit_use_double_quant=True` — quantize the quantization constants too (saves extra memory)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-v0.1"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading Mistral-7B in 4-bit quantization... (takes 3-5 mins)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"\n✅ Model loaded!")
print(f"Total parameters: {model.num_parameters():,}")

## 🔧 Step 5: Configure LoRA Adapters

### What is LoRA?
Instead of updating all 7.2B parameters during training, LoRA adds small **trainable adapter matrices** to specific layers. Only these adapters are trained — the original model weights stay frozen.

### Key Parameters:
- `r=16` — rank of the adapter matrices. Higher = more parameters, more capacity
- `lora_alpha=32` — scaling factor (typically 2x the rank)
- `target_modules` — which attention/MLP layers to apply LoRA to
- `lora_dropout=0.05` — dropout for regularization

### Result:
We train only **42M parameters out of 7.2B** — that's just **0.58%** of the model!
This is why LoRA fine-tuning is so memory-efficient.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,                    # adapter rank
    lora_alpha=32,           # scaling factor
    target_modules=[         # layers to apply LoRA
        "q_proj", "k_proj",
        "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Show how few parameters we're actually training
model.print_trainable_parameters()

## 📝 Step 6: Format the Dataset

We convert each dataset record into the **Alpaca prompt format** that Mistral understands:

```
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
<natural language description>

### Input:
<optional input>

### Output:
<python code>
```

**Why this format?**
Mistral-7B was pre-trained on data that includes this instruction format, so using it helps the model understand the task structure quickly.

**Training subset:**
We use 1,000 samples for this demo (free Colab has time limits). With more data and epochs, the model would generate better code.

In [ ]:
def format_prompt(example):
    return {
        "text": f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Output:
{example['output']}"""
    }

# Apply formatting
dataset_formatted = dataset.map(format_prompt)

# Use 1000 samples for free Colab (increase for better results)
train_dataset = dataset_formatted.select(range(1000))

print(f"Training samples: {len(train_dataset)}")
print(f"\nFormatted sample preview:")
print(train_dataset[0]['text'][:400])

## 🏋️ Step 7: Fine-tune the Model

We use HuggingFace's **SFTTrainer** (Supervised Fine-Tuning Trainer) which handles the training loop.

### Training Configuration:
- `num_train_epochs=1` — one full pass over the training data
- `per_device_train_batch_size=2` — 2 samples per GPU step
- `gradient_accumulation_steps=4` — accumulate gradients over 4 steps (effective batch size = 8)
- `learning_rate=2e-4` — standard learning rate for LoRA fine-tuning
- `report_to="wandb"` — log metrics to Weights & Biases

### What to expect:
- Training takes ~20-25 minutes on free T4 GPU
- Loss should decrease from ~0.85 → ~0.47 (model is learning!)
- Lower loss = model is better at predicting the next token in code

In [ ]:
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

sft_config = SFTConfig(
    output_dir="./mistral-code-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch size = 8
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    logging_steps=10,
    save_steps=100,
    report_to="wandb",
    run_name="mistral-code-lora",
    dataset_text_field="text",
    max_length=512,
)

# Clean dataset with only text field
formatted_data = [{"text": item["text"]} for item in train_dataset]
clean_dataset = Dataset.from_list(formatted_data)

trainer = SFTTrainer(
    model=model,
    train_dataset=clean_dataset,
    args=sft_config,
    processing_class=tokenizer,
)

print("🚀 Starting training...")
print("This will take ~20-25 minutes on free T4 GPU")
print("-" * 50)
trainer.train()
print("\n✅ Training complete!")

## 💾 Step 8: Save and Push to HuggingFace Hub

We save the **LoRA adapter weights** (not the full model!) to HuggingFace Hub.

### What gets saved?
Only the **42M adapter parameters** (~83MB), not the full 7.2B model. Anyone who wants to use this model loads the base Mistral-7B and applies our adapter on top.

### Why HuggingFace Hub?
- Free hosting for model weights
- Industry standard for sharing ML models
- Anyone can load and use your model with one line of code
- Shows up on your HuggingFace profile — great for your portfolio!

In [ ]:
REPO_NAME = "navnani12/mistral-code-lora"

print("Saving model locally...")
trainer.save_model("./mistral-code-finetuned")
tokenizer.save_pretrained("./mistral-code-finetuned")

print("Pushing LoRA adapter to HuggingFace Hub...")
model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)

print(f"\n✅ Model pushed successfully!")
print(f"🔗 View at: https://huggingface.co/{REPO_NAME}")

## 🧪 Step 9: Test the Fine-tuned Model

Now let's test what our model learned! We pass natural language instructions and see if it generates correct Python code.

### Inference Parameters:
- `max_new_tokens=150` — generate up to 150 tokens
- `temperature=0.1` — low temperature = more deterministic, focused output
- `repetition_penalty=1.3` — penalizes repeating the same tokens
- `do_sample=True` — use sampling instead of greedy decoding

### Expected Results:
With only 1000 training samples and 1 epoch, the model generates basic but functional Python code. Training on the full 18k dataset with 3 epochs would produce significantly better results.

In [ ]:
import torch

model.eval()  # set model to evaluation mode (disables dropout)

def generate_code(instruction):
    """
    Generate Python code from a natural language instruction.
    
    Args:
        instruction: Natural language description of what to code
    Returns:
        Generated Python code as a string
    """
    prompt = f"""### Instruction:
{instruction}

### Output:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.3,
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("### Output:")[-1].strip()


# Test with different instructions
test_cases = [
    "Write a Python function to check if a number is prime",
    "Write a Python function to reverse a string",
    "Write a Python function to find duplicates in a list",
]

for i, instruction in enumerate(test_cases, 1):
    print(f"Test {i}: {instruction}")
    print("-" * 50)
    print(generate_code(instruction))
    print()

## 📊 Results & Next Steps

### What we achieved:
- ✅ Fine-tuned a **7.2B parameter LLM** on a free GPU
- ✅ Used **QLoRA** to reduce memory from 28GB → 6GB
- ✅ Trained only **0.58% of parameters** (42M/7.2B) using LoRA
- ✅ Loss reduced from **0.865 → 0.47** during training
- ✅ Model generates working Python code from natural language
- ✅ Fine-tuned model published to HuggingFace Hub

### To improve results:
| Change | Expected Impact |
|--------|----------------|
| Train on all 18k samples (vs 1k) | Much better code quality |
| Increase epochs to 3 | Better convergence |
| Increase LoRA rank to 32 | More model capacity |
| Use Colab Pro (A100 GPU) | 3x faster training |

### How to use this model:
```python
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model = AutoModelForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.1")
model = PeftModel.from_pretrained(model, "navnani12/mistral-code-lora")
```